# Sprint 2 -- Modeling

**Goal:** predict all 731 binary `BIL_*` columns from the 745 binary `CRM_*` columns,
maximising the **Exact Match Ratio (EMR)** -- a row counts only if every one of its 731 bits
is right.

**Headline results**

| | EMR |
|---|---|
| Best k-NN baseline (k=15) | 51.98% (validation) |
| Label Powerset (hard ceiling 23.1%) | 23.09% (validation) |
| Section 11 pipeline as specified | 12.80% (validation) |
| Section 10 pipeline as specified | 50.58% (validation) |
| **Champion: regularised per-label LightGBM + 42 additive rules** | **87.84%** (validation, seed 42) / **86.49%** (independent split, seed 7) |
| Champion on the real test set (`solution.csv`, external diagnostic only) | **95.04%** |

Champion F1: micro **0.966** / macro **0.372** on validation; micro **0.998** / macro
**0.596** on test (full report with confusion matrices in sections 8 and 11).

**Structure.** 1. Setup -- 2. Validation design -- 3. Experiment tracker -- 4. Exp 1: lookup
and k-NN -- 5. Sprint 1 pipelines: Section 10 vs Section 11 -- 6. Modelling paradigms --
7. The variance finding -- 8. Champion: validation and error analysis -- 9. Robustness --
10. Final model and submission -- 11. External diagnostic -- 12. Conclusions.

Reusable code lives in `src/` (`data`, `validation`, `metrics`, `models`, `experiment`);
every experiment in the tracker is reproducible with the scripts in `scripts/`. The full lab
notes are in `docs/ITERATION_LOG.md` and the full table in `docs/EXPERIMENT_LEADERBOARD.md`.

## 1. Setup

In [ ]:
import os
import sys

os.environ.setdefault('OMP_NUM_THREADS', '6')
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '6')
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

import numpy as np
import pandas as pd
from IPython.display import display

from src import data, experiment as E, metrics as M, validation as V
from src.models import BinaryRelevanceLGBM
from scripts.exp1_lookup_knn import knn_hamming

pd.set_option('display.max_colwidth', 80)

`src/data.load()` reads `train.csv`, `test.csv` and the Sprint 1 exports once with an explicit
`uint8` dtype map (never pandas' default `int64`) and caches them as a compact `.npz` in
`data/cache/` (gitignored), so later runs load in seconds.

In [ ]:
d = data.load()
print('train CRM', d['X_train'].shape, d['X_train'].dtype, '| train BIL', d['Y_raw'].shape,
      '| test CRM', d['X_test'].shape)
print(f"memory: {(d['X_train'].nbytes + d['Y_raw'].nbytes + d['X_test'].nbytes) / 1e6:.0f} MB")

## 2. Validation design

Two facts from Sprint 1 drive the design:

1. **Repeated configurations leak.** 64% of training rows repeat another row's CRM
   configuration, so a random row split lets the model answer validation rows by
   memorisation (`validation_design.csv`). The split is therefore grouped on the **exact CRM
   configuration** (`np.packbits` row keys), so no configuration appears on both sides.
2. **Training labels contain provisioning errors, the test labels do not.** Validation rows
   are therefore scored against **consensus targets**: for each CRM configuration, the most
   frequent *complete* 731-bit BIL configuration among its rows. Taking the mode over whole
   rows (not per column) guarantees the target is a configuration that actually occurred.

In [ ]:
d = E.setup(seed=42)
tr, va = d['tr'], d['va']
print(f"{d['is_noisy'].sum():,} training rows ({d['is_noisy'].mean():.2%}) differ from their "
      "configuration's consensus")
print(f'train fold {len(tr):,} rows | validation fold {len(va):,} rows')

seen = set(d['groups'][tr].tolist())
train_keys = set(V.row_keys(d['X_train']))
print(f"validation rows whose CRM config is in the train fold: "
      f"{np.mean([g in seen for g in d['groups'][va]]):.2%}")
print(f"test rows whose CRM config is anywhere in train.csv: "
      f"{np.mean([k in train_keys for k in V.row_keys(d['X_test'])]):.4%}")

The grouped split reproduces the real test condition: essentially no test configuration
exists in `train.csv`, and no validation configuration exists in the training fold. Every
model has to **generalise to unseen CRM configurations**. This also means Sprint 1's
"exact-lookup ceiling" of 99.3% is a statistic *about* the training set, not something
achievable on test.

## 3. Experiment tracker

Every run logs to one results file through `src/experiment.log` (exported to
`data/derived/sprint2_experiment_results.csv`). All runs share the split and targets above
(seed 42) unless `split_seed` says otherwise.

In [ ]:
res = pd.read_csv('../data/derived/sprint2_experiment_results.csv')
res['val_emr_%'] = (res['emr'] * 100).round(2)
cols = ['exp_id', 'pipeline', 'family', 'params', 'val_emr_%', 'hamming_loss', 'train_time_s', 'notes']
display(res[cols])

## 4. Exp 1 -- exact lookup and Hamming k-NN

Exact lookup is inapplicable (0% coverage above). The k-NN below measures Hamming distance
between binary CRM vectors with one chunked matrix product
($d(a,b)=|a|+|b|-2\,a\cdot b$) against the unique training configurations, each labelled with
its consensus BIL configuration.

In [ ]:
Xu, Yu, cnt = E.dedup_configs(d['X_train'][tr], d['Y_cons'][tr])
idx, dist = knn_hamming(Xu, d['X_train'][va], k=15)
print(f'nearest training neighbour at Hamming distance <= 1 for {np.mean(dist[:, 0] <= 1):.1%} of val rows')

knn_rows = []
for k in (1, 15):
    if k == 1:
        pred = Yu[idx[:, 0]]
    else:
        w = (1 / (1 + dist[:, :k])) * np.log1p(cnt[idx[:, :k]])
        pred = ((Yu[idx[:, :k]] * w[:, :, None]).sum(1) / w.sum(1, keepdims=True) >= 0.5).astype(np.uint8)
    knn_rows.append({'k': k, **E.score(d, pred)})
display(pd.DataFrame(knn_rows).round(4))

**The key structural finding of Sprint 2.** For 70% of validation rows the nearest training
configuration differs by a single CRM pack -- yet copying its billing gives almost 0% EMR,
and ~59% of those copies are wrong by **exactly one BIL bit**. Changing one CRM pack changes
one BIL bit: the CRM -> BIL mapping is close to **additive, pack by pack**. Neighbour copying
cannot express that; voting over 15 neighbours partly averages it away (52%). A model that
learns each BIL bit as a function of the CRM packs is the natural fit.

## 5. Sprint 1 pipelines: Section 10 vs Section 11

Each pipeline was first run **as Sprint 1 defined it** (Section 10: all rows with majority-
relabelled targets; Section 11: all rows, raw targets, 742 CRM + 45 SVD features, 42 rule
columns). Then, to separate the *feature* effect from the *training-row* effect, every
feature set was trained on **identical rows**: unique configurations with consensus labels.
Model: one LightGBM per label (binary relevance), 100 trees, 15 leaves.

In [ ]:
pick = ['E3b', 'E3c', 'E3c+rules', 'E3a', 'E3a2', 'E3a2+rules', 'E3d', 'E3e', 'E3e+rules']
display(res.set_index('exp_id').loc[pick, ['pipeline', 'family', 'val_emr_%', 'hamming_loss', 'notes']])

- **Training rows matter first.** Training on every row (E3b, E3c) or weighting unique
  configurations by their multiplicity (E3a, weights up to 8,114) lets a few very frequent
  configurations dominate the trees, and rare labels start firing on ~5% of rows (e.g.
  `BIL_4420_PACK`, true prevalence 0.12%). Unique configurations, unweighted: **80.05%**.
- **Section 10's features cost ~6 points** (74.31%). Collapsing CRM columns with |r| >= 0.95
  deletes packs that each drive a *different* BIL pack, which Sprint 1 section 6b had flagged.
- **Section 11's features cost ~33 points** (47.42%). Its CRM subset differs from the raw one by
  3 near-empty columns, so the damage comes from the **45 SVD components**: continuous,
  configuration-specific values that trees split on to memorise configurations.
- **The 42 Section 11 rules are useful as post-processing** (+0.3 points on a good model), but
  none is exact on training data -- applied to every training row they agree with consensus on
  only 94.4% of rows, so they are an override on top of a model, not a replacement for it.

**Decision:** keep Sprint 1's *analysis* (noise/consensus handling, validation design, the
rules) but feed the models the **raw 745 binary CRM columns**.

## 6. Modelling paradigms

All four paradigms from the brief, on the same split:

| Paradigm | Implementation |
|---|---|
| Independent binary relevance | `BinaryRelevanceLGBM` -- 731 LightGBM models, trained in parallel threads |
| Multi-output trees / neighbours | Hamming k-NN (section 4): one joint prediction of the whole vector |
| Classifier chain | `CoOccurrenceChainLGBM` -- labels in prevalence order; each sees X plus its 30 most co-occurring upstream labels |
| Label powerset | Max-likelihood decoder over all 53k observed training configurations |

A multi-class model over 53k configurations is not trainable, and the label-powerset ceiling
explains why it would not matter anyway:

In [ ]:
ktr = pd.Series(V.row_keys(d['Y_cons'][tr])).value_counts()
kva = V.row_keys(d['Y_cons'][va])
seen_cfg = set(ktr.index)
print(f'val rows whose BIL configuration exists in the train fold: {np.mean([k in seen_cfg for k in kva]):.2%}')
for K in (100, 1000, 5000):
    top = set(ktr.index[:K])
    print(f'  ...covered by the top-{K} training configurations: {np.mean([k in top for k in kva]):.2%}')

pick = ['E1-knn15', 'E2-lp', 'E2-hyb1.0', 'E3a2', 'E4-cc30']
display(res.set_index('exp_id').loc[pick].drop_duplicates()[['family', 'val_emr_%', 'hamming_loss', 'notes']])

Unseen CRM configurations mostly produce **unseen BIL configurations**, so label powerset is
capped at ~23% by construction -- and the decoder reaches 23.09%, essentially the whole
ceiling. Snapping only near-certain rows to a seen configuration leaves binary relevance
unchanged. The classifier chain adds +0.5 points over plain binary relevance.

## 7. The variance finding

Averaging binary-relevance and chain probabilities gave 78.1% at threshold 0.5 but 87.2% at
0.6 -- effectively an OR vs. an AND of the two models. Checked directly: where the two models
disagree (60k cells, 24% of rows) the true bit is **0 in 97.6% of cases**. Each model emits
many *confident, idiosyncratic false positives* the other does not: a variance problem.
The principled fix is to regularise the per-label models rather than to AND them together.

In [ ]:
pick = ['E5-avg-t0.5', 'E5-avg-t0.6', 'E5-bag3-t0.5', 'E5-reg-mcs50-t0.5', 'E5-reg-mcs20-t0.5',
        'E5-reg-mcs20-t0.6', 'E4-cc30-mcs20', 'E5-avg-mcs20-t0.5']
display(res.set_index('exp_id').loc[pick, ['family', 'params', 'val_emr_%', 'hamming_loss', 'train_time_s']])

Raising `min_child_samples` from 5 to 20 (with `reg_lambda=1`) makes a **single** binary-
relevance model beat the AND ensemble (87.84%) at the default threshold 0.5. Bagging three
less-regularised models is 3x slower and worse. Once regularised, the chain and the blend no
longer help: the label dependencies the chain captured were mostly compensating for
overfitting. **The simplest model wins.**

## 8. Champion: validation run and error analysis

Champion: per-label LightGBM on the raw 745 CRM columns, trained on unique CRM configurations
with consensus labels, `min_child_samples=20`, `reg_lambda=1`, threshold 0.5, then the 42
Section 11 additive rules. Re-trained here on the train fold (~3.5 minutes).

In [ ]:
CHAMPION = dict(min_child_samples=20, reg_lambda=1.0)
with E.Timer() as t:
    champ = BinaryRelevanceLGBM(CHAMPION).fit(Xu, Yu)
P_val = champ.predict_proba(d['X_train'][va])
pred_val = data.apply_additive_rules((P_val >= 0.5).astype(np.uint8), d['X_train'][va], d['crm_cols'], d['bil_cols'])
print(f'fit {t.s:.0f}s')
print({k: round(v, 5) for k, v in E.score(d, pred_val).items()})

### Full metric report (validation)

EMR is the competition metric; the others explain *how* the model is right or wrong.
The confusion matrix counts cells (one row x one BIL column). Macro F1 averages over BIL
columns with at least one positive, giving each column equal weight however rare it is.

In [ ]:
Yv = d['Y_cons'][va]
val_summary, val_cm, val_f1 = M.multilabel_report(Yv, pred_val, P_val)
display(val_summary.to_frame('validation').style.format('{:.5f}'))
display(val_cm.style.format('{:,}'))
print(f'false negatives per false positive: {val_cm.iloc[0, 1] / val_cm.iloc[1, 0]:.1f}')
print(f'per-label F1 over {len(val_f1)} columns: median {np.median(val_f1):.3f} | '
      f'F1 = 1: {(val_f1 == 1).sum()} | F1 < 0.8: {(val_f1 < 0.8).sum()}')

- **Micro vs macro F1 tell different stories.** Micro F1 (0.97) is dominated by the common
  columns, which are almost always right. Macro F1 (0.37) weights all 731 columns equally,
  and 607 of them are active in under 1% of rows; many of those rare columns are the ones
  that fail in the rare-pack rows below.
- **The model misses positives far more than it invents them** (~7 false negatives per
  false positive; precision 0.99, recall 0.94). That is the other side of the regularisation
  in section 7: it removed the confident false positives that broke rows, but it
  under-predicts rare billing packs. Lowering the threshold only on rare columns is the
  obvious Sprint 3 lever.

In [ ]:
Yv = d['Y_cons'][va]
err = M.row_errors(Yv, pred_val)
print('wrong bits per validation row:', pd.Series(np.minimum(err, 6)).value_counts().sort_index()
      .rename(index={6: '6+'}).to_dict())
print('\ncolumns that alone break an otherwise perfect row:')
display(M.top_offending_columns(Yv, pred_val, d['bil_cols'], top_n=10).round(4))

Most remaining validation failures are **catastrophic rows** with many wrong bits (about as
many false positives as false negatives: the right number of packs, the wrong identities).
They are not label noise -- Sprint 1 measured provisioning-noise variants at a median of
1 bit (max 10). They come from **moderately rare CRM packs**:

In [ ]:
freq = d['X_train'][tr].sum(0)
rare = (freq > 0) & (freq < 200)
has_rare = d['X_train'][va][:, rare].any(1)
tbl = pd.DataFrame({
    'val rows': [has_rare.sum(), (~has_rare).sum()],
    'EMR': [(err[has_rare] == 0).mean(), (err[~has_rare] == 0).mean()],
    'share with 6+ wrong bits': [(err[has_rare] >= 6).mean(), (err[~has_rare] >= 6).mean()],
}, index=['contains a CRM pack with <200 train rows', 'all packs frequent'])
display(tbl.round(4))
print(f'{rare.sum()} such CRM packs | mean per row: validation {d["X_train"][va][:, rare].sum(1).mean():.3f}, '
      f'test {d["X_test"][:, rare].sum(1).mean():.3f}')

These rare packs have no dominant BIL partner (e.g. `CRM_1761_PACK` co-occurs with its best
BIL match in only 10.8% of its rows), so their billing depends on combinations of packs seen
too rarely to learn, and the errors stack within a row. **The test set carries about 11x
fewer of these packs per row than the validation fold**, so validation EMR should understate
test EMR -- which section 11 confirms.

## 9. Robustness: an independent split

In [ ]:
pick = ['E5-reg-mcs20-t0.5', 'E5-reg-mcs20-t0.6', 'E5-reg-mcs20-s7-t0.5', 'E5-reg-mcs20-s7-t0.6']
display(res.set_index('exp_id').loc[pick, ['params', 'val_emr_%', 'hamming_loss']])

On a second, independent grouped split (seed 7) the champion scores 86.49%. The ranking of
choices is the same on both splits (threshold 0.5 > 0.6; rules help by ~0.2 points), so the
selection was not an artefact of one validation fold.

## 10. Final model and submission

The frozen champion is refit on **all** 67,434 unique CRM configurations of `train.csv`
(consensus labels) and applied to `test.csv` (~3.5 minutes).

In [ ]:
d_all = data.load()
g_all, _ = V.factorize_rows(d_all['X_train'])
Y_cons_all, _ = V.consensus_targets(g_all, d_all['Y_raw'])
Xa, Ya, _ = E.dedup_configs(d_all['X_train'], Y_cons_all)
with E.Timer() as t:
    final = BinaryRelevanceLGBM(CHAMPION).fit(Xa, Ya)
P_test = final.predict_proba(d_all['X_test'])
pred_test = data.apply_additive_rules((P_test >= 0.5).astype(np.uint8), d_all['X_test'],
                                      d_all['crm_cols'], d_all['bil_cols'])
print(f'trained on {len(Xa):,} configurations in {t.s:.0f}s')

sub = pd.DataFrame({'MSISDN': d_all['msisdn_test'],
                    'Bill_Conf': [''.join(r) for r in pred_test.astype(str)]})
assert len(sub) == 97_100 and sub.notna().all().all() and sub['MSISDN'].is_unique
assert sub['Bill_Conf'].str.len().eq(731).all() and sub['Bill_Conf'].str.fullmatch('[01]+').all()
sub.to_csv('../data/derived/submission_sprint2.csv', index=False)
print('submission_sprint2.csv written:', sub.shape, '| Bill_Conf length 731 | no NaNs')
sub.head()

## 11. External diagnostic: `solution.csv`

`solution.csv` appears to be the test answer key. It was **not** used for training, feature
engineering, or any model or threshold choice; model selection was frozen (section 9) before
this was computed. It is reported only as a diagnostic, pending confirmation from the
instructor on how it may be used.

In [ ]:
sol = pd.read_csv('../data/solution.csv', dtype=str)
m = sub.merge(sol, on='MSISDN', suffixes=('', '_true'))
assert len(m) == len(sub)
to_bits = lambda s: np.array([np.frombuffer(x.encode(), np.uint8) - 48 for x in s])
Y_true, Y_pred = to_bits(m['Bill_Conf_true']), to_bits(m['Bill_Conf'])
print({k: round(v, 5) for k, v in M.evaluate(Y_true, Y_pred).items()})

In [ ]:
# sub (and therefore m) keeps test.csv row order, so P_test rows line up with Y_true
test_summary, test_cm, test_f1 = M.multilabel_report(Y_true, Y_pred, P_test)
display(pd.concat([val_summary.rename('validation'), test_summary.rename('test (solution.csv)')], axis=1)
        .style.format('{:.5f}'))
display(test_cm.style.format('{:,}'))
err_t = M.row_errors(Y_true, Y_pred)
print('test rows by number of wrong bits:', pd.Series(np.minimum(err_t, 6)).value_counts().sort_index()
      .rename(index={6: '6+'}).to_dict())
print(f'per-label F1 over {len(test_f1)} columns with positives: median {np.median(test_f1):.3f} | '
      f'F1 = 1: {(test_f1 == 1).sum()} | F1 < 0.8: {(test_f1 < 0.8).sum()}')

Test EMR is well above validation, as the rare-pack analysis predicted: the test set has far
fewer of the packs that cause catastrophic rows, and its labels are clean. On test, nearly
all remaining errors are single bits, micro F1 is 0.998, and the median per-column F1 rises
from 0.23 to 0.97. Macro F1 stays lower (0.60) because a minority of rare columns is still
under-predicted -- again more false negatives than false positives.

## 12. Conclusions and Sprint 3 directions

1. **The mapping is close to additive per pack**, so per-label models beat neighbour copying
   (80-88% vs 52%), and label powerset is structurally capped (~23%) because unseen CRM
   configurations produce unseen BIL configurations.
2. **How training rows are weighted matters more than the model family.** Deduplicating to
   unique configurations with consensus labels, *unweighted*, took the same model from 58.7%
   to 80.1%.
3. **Neither Sprint 1 feature transformation helps**: Section 10's column collapsing costs
   ~6 points, Section 11's SVD components ~33. Sprint 1's noise analysis, validation design
   and additive rules are what carry over.
4. **The dominant error was variance**, not missing label dependencies: regularising
   (`min_child_samples=20`) gave +7.8 points and made chains and blends unnecessary.
5. Champion: **87.84% / 86.49%** validation EMR on two independent grouped splits,
   **95.04%** on the test set (external diagnostic).

**For Sprint 3 (optimisation and explainability):** tune the regularisation and per-label
thresholds against grouped-CV EMR; target the rare-pack rows (e.g. pack-interaction features,
or pooling evidence across rare packs); and use SHAP to verify each BIL column relies on the
CRM packs that should drive it rather than on noise in the training labels.